# MangoCare AI · Anthracnose vs. Healthy

Train a standalone YOLO11n image classifier. Each input should show one mango leaf and receives one image-level prediction.

1. Upload `Classification/prepared/MangoCare_Classification_Dataset.zip` to **My Drive → MangoCare**.
2. Upload this notebook to Colab. Choose **Runtime → Change runtime type → GPU**, then run the cells in order and authorize Drive.
3. Review the test confusion matrix and per-class recall before using the checkpoint on new photos.

The prepared dataset removes exact pixel duplicates and keeps likely related images together when splitting. The Anthracnose class has few independent examples, so this run is a baseline. Photos gathered outside these folders are needed to assess performance on new sources.

For the final model, run the setup and staging cells, then jump to **Final model**. The baseline section is optional.

In [ ]:
%pip install -q ultralytics==8.3.223

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab before training.")
torch.zeros(1, device="cuda:0").sum().item()
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Stage the prepared dataset

The ZIP stays in Drive. This cell extracts the split folders into temporary local Colab storage and verifies the two class folders in each split.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

from google.colab import drive

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/MangoCare")
zip_path = drive_root / "MangoCare_Classification_Dataset.zip"
if not zip_path.is_file():
    raise FileNotFoundError(f"Upload the prepared ZIP to {zip_path}")

work_dir = Path(mkdtemp(prefix="mangocare-classification-", dir="/content"))
dataset = work_dir / "dataset"
labels = {0: "Anthracnose", 1: "Healthy"}
splits = {"train", "val", "test"}

with ZipFile(zip_path) as archive:
    for member in archive.infolist():
        parts = Path(member.filename).parts
        if (len(parts) != 3 or parts[0] not in splits or parts[1] not in labels.values()
                or Path(parts[2]).suffix.lower() != ".jpg"):
            raise ValueError(f"Unexpected path in prepared ZIP: {member.filename}")
    archive.extractall(dataset)

for split in ("train", "val", "test"):
    counts = {label: len(list((dataset / split / label).glob("*.jpg"))) for label in labels.values()}
    if not all(counts.values()):
        raise ValueError(f"Missing images in {split}: {counts}")
    print(split, counts)

## Train the classifier

Start with pretrained `yolo11n-cls.pt`, 224 × 224 inputs, batch size 16, and a maximum of 25 epochs. Validation selects the best checkpoint. If CUDA runs out of memory, reduce `batch` to 8 and rerun this cell. The best classifier is copied to a new folder in Drive immediately after training.

In [ ]:
import shutil
from datetime import datetime, timezone
from uuid import uuid4

image_size = 224
batch = 16
classifier = YOLO("yolo11n-cls.pt", task="classify")
classifier.train(
    data=str(dataset), imgsz=image_size, batch=batch, epochs=25, patience=7,
    device=0, workers=2, seed=42, deterministic=True, cache=False,
    project=str(work_dir / "runs"), name="binary_leaf_classifier",
)
checkpoint = Path(classifier.trainer.best)
if not checkpoint.is_file():
    raise FileNotFoundError(f"Training did not write {checkpoint}")
best = YOLO(str(checkpoint), task="classify")
if best.task != "classify" or best.names != labels:
    raise ValueError(f"Unexpected classifier metadata: {best.task}, {best.names}")

run_name = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8]
drive_run = drive_root / "models" / "classification" / run_name
drive_run.mkdir(parents=True)
shutil.copy2(checkpoint, drive_run / "best.pt")
print(f"Classifier saved to {drive_run / 'best.pt'}")

## Evaluate the held-out test split

Report top-1 accuracy, recall by class, balanced accuracy, and the raw confusion matrix. Read the support counts alongside percentages. A single error can move the Anthracnose recall substantially.

In [ ]:
import json

test_dir = work_dir / "evaluation" / "test"
metrics = best.val(
    data=str(dataset), split="test", imgsz=image_size, batch=batch,
    device=0, workers=2, plots=True, project=str(test_dir.parent), name=test_dir.name,
    exist_ok=True,
)
matrix = metrics.confusion_matrix.matrix.astype(int)
support = matrix.sum(axis=0)
if any(count == 0 for count in support):
    raise ValueError("Test evaluation did not include both classes.")
recalls = {label: float(matrix[idx, idx] / support[idx]) for idx, label in labels.items()}
summary = {
    "class_ids": labels,
    "top1_accuracy": float(metrics.top1),
    "balanced_accuracy": sum(recalls.values()) / len(recalls),
    "per_class_recall": recalls,
    "test_images_per_class": {label: int(support[idx]) for idx, label in labels.items()},
    "confusion_matrix_rows_predicted_columns_true": matrix.tolist(),
}
(drive_run / "test_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
shutil.copytree(test_dir, drive_run / "test_evaluation", dirs_exist_ok=True)
print(json.dumps(summary, indent=2))

## Final model Â· train on all unique photos

Run this cell after **Stage the prepared dataset**. The earlier baseline cells are optional. This stage trains a fresh classifier on all 91 prepared photos, including the former validation and test photos. It saves the last checkpoint after 25 epochs. Any validation scores printed during this stage are internal diagnostics because those photos also appear in training. Use new photos for an independent test of this final model.

Download `mangocare_classifier.pt` when training finishes. This checkpoint classifies a single leaf photo and uses a separate filename from the object detector.

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from uuid import uuid4

final_work = Path(mkdtemp(prefix="final-classifier-", dir=work_dir))
final_dataset = final_work / "dataset"
for label in labels.values():
    target = final_dataset / "train" / label
    target.mkdir(parents=True)
    for split in ("train", "val", "test"):
        for photo in sorted((dataset / split / label).glob("*.jpg")):
            destination = target / photo.name
            if destination.exists():
                raise ValueError(f"Repeated filename across splits: {photo.name}")
            shutil.copy2(photo, destination)
    print(f"{label}: {len(list(target.glob('*.jpg')))} final training photos")

# Ultralytics requires a validation directory and can evaluate at the final epoch.
# These already-trained-on photos are only for internal diagnostics.
shutil.copytree(dataset / "val", final_dataset / "val")

final_epochs = 25
final_batch = 16
final_image_size = 224
final_model = YOLO("yolo11n-cls.pt", task="classify")
final_model.train(
    data=str(final_dataset), imgsz=final_image_size, batch=final_batch,
    epochs=final_epochs, patience=0, val=False, device=0, workers=2,
    seed=42, deterministic=True, cache=False, plots=False,
    project=str(final_work / "runs"), name="all_photos",
)
last = Path(final_model.trainer.last)
if not last.is_file():
    raise FileNotFoundError(f"Final training did not write {last}")
trained = YOLO(str(last))
if trained.task != "classify" or trained.names != labels:
    raise ValueError(f"Unexpected classifier metadata: {trained.task}, {trained.names}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
final_drive = drive_root / "models" / "classification" / f"final-{stamp}-{uuid4().hex[:8]}"
final_drive.mkdir(parents=True)
final_checkpoint = final_drive / "mangocare_classifier.pt"
shutil.copy2(last, final_checkpoint)
metadata = {
    "task": "classify",
    "class_ids": labels,
    "image_size": final_image_size,
    "epochs": final_epochs,
    "training_images": {
        label: len(list((final_dataset / "train" / label).glob("*.jpg")))
        for label in labels.values()
    },
    "independent_evaluation": False,
}
(final_drive / "classifier_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Final classifier saved to {final_checkpoint}")